In [0]:

from __future__ import annotations

import argparse
import json

from inference_artifacts import (
    fetch_forecast_weather_rows,
    get_base_path,
    parse_positive_int_env,
    parse_request_timestamp,
    read_step_artifact,
    resolve_inference_run_context,
    write_step_artifact,
)


def run_weather_step(
    run_id: str,
    run_ts: str,
    horizon_steps: int,
) -> tuple[dict, str]:
    base_path = get_base_path()

    live_payload = read_step_artifact(
        base_path=base_path,
        run_id=run_id,
        step_name="live_station",
        filename="live_station.json",
    )

    request_ts = parse_request_timestamp(live_payload["request_timestamp"])
    live_station = live_payload["live_station"]

    weather_horizon = fetch_forecast_weather_rows(
        lat=float(live_station["lat"]),
        lon=float(live_station["lon"]),
        request_ts_utc=request_ts,
        horizon_steps=horizon_steps,
    )

    payload = {
        "run_id": run_id,
        "run_ts": run_ts,
        "request_timestamp": request_ts.isoformat().replace("+00:00", "Z"),
        "station_id": live_station["station_id"],
        "canonical_station_id": live_payload["canonical_station_id"],
        "horizon_steps": horizon_steps,
        "rows": weather_horizon.rows,
        "fallback_applied": weather_horizon.fallback_applied,
        "missing_timestamps": weather_horizon.missing_timestamps,
    }

    artifact_path = write_step_artifact(
        base_path=base_path,
        run_id=run_id,
        step_name="weather",
        payload=payload,
        filename="weather.json",
    )
    return payload, artifact_path


def main() -> None:
    parser = argparse.ArgumentParser(description="Inference Step 02: weather horizon")
    parser.add_argument("--run-id", default=None)
    parser.add_argument("--run-ts", default=None)
    parser.add_argument(
        "--horizon-steps",
        type=int,
        default=None,
        help="Optional horizon override. Defaults to INFERENCE_HORIZON_STEPS or 6.",
    )
    args = parser.parse_args()

    run_id, run_ts = resolve_inference_run_context(args.run_id, args.run_ts)
    horizon_steps = args.horizon_steps or parse_positive_int_env(
        "INFERENCE_HORIZON_STEPS",
        default=6,
    )

    payload, artifact_path = run_weather_step(
        run_id=run_id,
        run_ts=run_ts,
        horizon_steps=horizon_steps,
    )

    print(
        json.dumps(
            {
                "step": "weather",
                "run_id": run_id,
                "run_ts": run_ts,
                "artifact_path": artifact_path,
                "row_count": len(payload["rows"]),
                "fallback_applied": payload["fallback_applied"],
            },
            indent=2,
        )
    )

In [0]:
import os
def store_databricks_params_as_env():
    if os.getenv("DATABRICKS_RUNTIME_VERSION"):
        # 1. Retrieve all parameters (widgets) as a dictionary
        # This returns a dict of { "parameter_name": "parameter_value" }
        all_params = dbutils.widgets.getAll()

        # 2. Store them in os.environ
        for key, value in all_params.items():
            os.environ[key] = str(value)

        print(f"Stored {len(all_params)} parameters in environment variables.")

def build_default_run_id() -> str:
    env_run_id = os.environ.get("INFERENCE_RUN_ID")
    if env_run_id and env_run_id.strip():
        return env_run_id.strip()

    pipeline_run_id = os.environ.get("PIPELINE_RUN_ID")
    if pipeline_run_id and pipeline_run_id.strip():
        return pipeline_run_id.strip()

    job_run_id = os.environ.get("PIPELINE_JOB_RUN_ID")
    if job_run_id and job_run_id.strip():
        repair_count = os.environ.get("PIPELINE_REPAIR_COUNT", "0").strip() or "0"
        return f"job_{job_run_id.strip()}_repair_{repair_count}"

    now_utc = datetime.datetime.now(timezone.utc)
    return f"run_{now_utc.strftime('%Y%m%d%H%M%S')}_{uuid.uuid4().hex[:8]}"


def build_default_run_ts() -> str:
    env_run_ts = os.environ.get("INFERENCE_RUN_TS")
    if env_run_ts and env_run_ts.strip():
        return env_run_ts.strip()

    pipeline_run_ts = os.environ.get("PIPELINE_RUN_TS")
    if pipeline_run_ts and pipeline_run_ts.strip():
        return pipeline_run_ts.strip()

    return datetime.datetime.now(timezone.utc).isoformat().replace("+00:00", "Z")

def is_job():
    try:
        # Access the internal notebook context
        context_json = dbutils.notebook.entry_point.getDbutils().notebook().getContext().toJson()
        context = json.loads(context_json)
        
        # Check for the existence of jobId in tags
        return "jobId" in context.get("tags", {})
    except:
        return False

import datetime
from datetime import timedelta, timezone
import uuid

if is_job():
    print("Running as a Databricks Job")
else:
    print("Running interactively (Manual)")
    now_utc = datetime.datetime.now(timezone.utc)
    dbutils.widgets.text("INFERENCE_REQUEST_JSON", '{"name":"du Mont-Royal / Clark","lat":45.51941,"lon":-73.58685,"station_id":"218"}')
    dbutils.widgets.text("INFERENCE_RUN_ID", "20260420080036_e9483314") #f"{now_utc.strftime('%Y%m%d%H%M%S')}_{uuid.uuid4().hex[:8]}")
    dbutils.widgets.text("INFERENCE_HORIZON_STEPS", "3")

In [0]:
store_databricks_params_as_env()

request_json = os.getenv("INFERENCE_REQUEST_JSON")
run_id = os.getenv("INFERENCE_RUN_ID")
horizon_steps = os.getenv("INFERENCE_HORIZON_STEPS")

print("args request json:", request_json)
print("args horizon_steps id:", horizon_steps)
print("args run_id:", run_id)

In [0]:
run_id, run_ts = resolve_inference_run_context(run_id, build_default_run_ts())

horizon_steps = parse_positive_int_env(
    "INFERENCE_HORIZON_STEPS",
    default=6,
)

payload, artifact_path = run_weather_step(
    run_id=run_id,
    run_ts=run_ts,
    horizon_steps=horizon_steps,
)

print(
    json.dumps(
        {
            "step": "weather",
            "run_id": run_id,
            "run_ts": run_ts,
            "artifact_path": artifact_path,
            "row_count": len(payload["rows"]),
            "fallback_applied": payload["fallback_applied"],
        },
        indent=2,
    )
)